# ORI-C Notebook 03 — Robustness Analysis

This notebook is a **secondary** robustness analysis. It explores how the threshold detection
verdict varies across plausible parameter alternatives.

> **Important:** Robustness analyses are secondary — they inform but **never change** the primary
> hypothesis verdicts derived from the pre-registered parameters.
> See `ORIC_POINT_OF_TRUTH.md` and `02_Protocol/DECISION_RULES_v2.md`.

We sweep over:
- Threshold sensitivity `k` ∈ {2.0, 2.5, 3.0} (default: 2.5)
- Consecutive steps `m` ∈ {2, 3, 4} (default: 3)
- Baseline window `baseline_n` ∈ {20, 30, 50} (default: 30)

---
**Setup:** `conda activate cumulative_symbolic` from repo root.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / '04_Code'))

from pipeline.ori_c_pipeline import ORICConfig, run_oric

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Imports OK')

## 1. Generate reference run (pre-registered parameters)

In [ ]:
SEED = 42
N_STEPS = 300

# Pre-registered defaults
K_DEFAULT = 2.5
M_DEFAULT = 3
BL_DEFAULT = 30

cfg = ORICConfig(seed=SEED, n_steps=N_STEPS, intervention='symbolic_injection')
df  = run_oric(cfg)
print(f'Run shape: {df.shape}')

## 2. Threshold detection helper

In [ ]:
def detect_threshold(dc, k, m, baseline_n):
    """Returns (threshold_value, first_crossing_index_or_None)."""
    baseline = dc[:baseline_n]
    mu, sigma = np.nanmean(baseline), np.nanstd(baseline)
    if sigma == 0:
        return np.nan, None
    thr = mu + k * sigma
    count = 0
    for i, v in enumerate(dc):
        if np.isfinite(v) and v > thr:
            count += 1
            if count >= m:
                return thr, i - m + 1
        else:
            count = 0
    return thr, None

dc = df['delta_C'].to_numpy(float)
thr_ref, idx_ref = detect_threshold(dc, K_DEFAULT, M_DEFAULT, BL_DEFAULT)
print(f'Reference: k={K_DEFAULT}, m={M_DEFAULT}, baseline_n={BL_DEFAULT}')
print(f'  → threshold={thr_ref:.4f}, crossing at step {idx_ref}')

## 3. Sweep: k and m (2D heatmap)

This is the standard robustness check: does the threshold detection verdict hold across
plausible parameter variants?

In [ ]:
k_vals  = [1.5, 2.0, 2.5, 3.0, 3.5]
m_vals  = [2, 3, 4, 5]
bl_vals = [20, 30, 50]

# Fix baseline_n = default, sweep k x m
results_km = np.zeros((len(k_vals), len(m_vals)))

for i, k in enumerate(k_vals):
    for j, m in enumerate(m_vals):
        _, idx = detect_threshold(dc, k, m, BL_DEFAULT)
        results_km[i, j] = idx if idx is not None else np.nan

# Plot heatmap
fig, ax = plt.subplots(figsize=(7, 5))

# Detected = any value, Not detected = NaN → mask
detected = ~np.isnan(results_km)
cmap = mcolors.ListedColormap(['#fee2e2', '#bbf7d0'])
im = ax.imshow(detected.astype(float), cmap=cmap, aspect='auto',
               vmin=0, vmax=1, origin='upper')

for i in range(len(k_vals)):
    for j in range(len(m_vals)):
        val = results_km[i, j]
        txt = f't={val:.0f}' if not np.isnan(val) else 'ND'
        ax.text(j, i, txt, ha='center', va='center',
                fontsize=9, color='#1e293b' if not np.isnan(val) else '#dc2626',
                fontweight='bold' if (k_vals[i] == K_DEFAULT and m_vals[j] == M_DEFAULT) else 'normal')

ax.set_xticks(range(len(m_vals)));  ax.set_xticklabels([f'm={m}' for m in m_vals])
ax.set_yticks(range(len(k_vals)));  ax.set_yticklabels([f'k={k}' for k in k_vals])
ax.set_title('Robustness sweep: threshold crossing step\n(green=detected, red=not detected)')

# Highlight pre-registered default
i0 = k_vals.index(K_DEFAULT)
j0 = m_vals.index(M_DEFAULT)
rect = plt.Rectangle((j0-.5, i0-.5), 1, 1, fill=False, edgecolor='black', lw=2.5)
ax.add_patch(rect)
ax.text(j0+.35, i0-.35, '★ pre-reg', fontsize=7, color='black')

plt.tight_layout()
plt.show()

## 4. Sweep: baseline_n (robustness to baseline window)

In [ ]:
bl_range = list(range(10, 80, 5))
crossings = []

for bl in bl_range:
    _, idx = detect_threshold(dc, K_DEFAULT, M_DEFAULT, bl)
    crossings.append(idx if idx is not None else np.nan)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(bl_range, crossings, 'o-', color='#2563eb', lw=1.5, ms=5)
ax.axvline(BL_DEFAULT, color='#dc2626', lw=1.5, ls='--', label=f'Default baseline_n={BL_DEFAULT}')
ax.set_xlabel('baseline_n')
ax.set_ylabel('Threshold crossing step')
ax.set_title('Robustness to baseline window size (k=2.5, m=3)')
ax.legend()
plt.tight_layout()
plt.show()

n_detected = sum(1 for c in crossings if not np.isnan(c))
print(f'Detected in {n_detected}/{len(bl_range)} baseline_n variants ({100*n_detected/len(bl_range):.0f}%)')

## 5. Run the official robustness script

For a full robustness report including alternative Cap specifications, use the pipeline script.

In [ ]:
import subprocess

OUT_ROBUST = REPO_ROOT / '05_Results/nb03_robustness'
OUT_ROBUST.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(REPO_ROOT / '04_Code/pipeline/run_robustness.py'),
    '--input',  str(REPO_ROOT / '03_Data/synthetic/synthetic_with_transition.csv'),
    '--outdir', str(OUT_ROBUST),
]

print('Running robustness script...')
r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO_ROOT))
print(r.stdout[-2000:] if len(r.stdout) > 2000 else r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-800:])

In [ ]:
# Read robustness results if available
csv_path = OUT_ROBUST / 'tables/robustness_results.csv'
if csv_path.exists():
    df_robust = pd.read_csv(csv_path)
    print(f'Robustness results: {df_robust.shape}')
    df_robust.head(10)
else:
    print('Robustness results not found — the run may use a different output structure.')
    # List what was written
    for p in sorted(OUT_ROBUST.rglob('*')):
        if p.is_file():
            print(' ', p.relative_to(REPO_ROOT))

## 6. Interpretation

**Robustness criteria (secondary, informative only):**

- If the threshold is detected across ≥ 75% of parameter variants → result is robust
- If detection is sensitive to a specific parameter → that parameter should be flagged as a limitation in the manuscript
- **Robustness analysis cannot change the primary verdict** — only the pre-registered parameters (k=2.5, m=3, α=0.01) determine the primary verdict

See `02_Protocol/DECISION_RULES_v2.md` for the full decision framework.

---
**Back to basics:** `notebook_01_synthetic_demo.ipynb` | `notebook_02_real_data_pilot.ipynb`